# Building Footprint Extraction using U-Net
Objective: Extract building footprints from high-resolution satellite imagery using a custom convolutional neural network (CNN).

### 1. Environment & Dependencies
Installing the necessary deep learning (PyTorch) and data processing tools.


In [ ]:
!pip install torch torchvision rasterio geopandas shapely opencv-python-headless matplotlib datasets

### 2. Cloud Data Pipeline & RAM Caching
Working with massive geospatial datasets (like MapAI) locally can cause hardware bottlenecks. To optimize performance, we stream the dataset directly from the cloud and cache a training subset into RAM. This allows for rapid, iterative batch processing without continuous network latency.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from datasets import load_dataset
import matplotlib.pyplot as plt

class CachedBuildingDataset(Dataset):
    def __init__(self, hf_dataset, num_samples=20):
        print(f"Downloading and caching {num_samples} images to RAM...")
        self.samples = []
        dataset_iter = iter(hf_dataset)
        
        transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Resize((256, 256), antialias=True)
        ])
        
        for _ in range(num_samples):
            sample = next(dataset_iter)
            image = transform(sample['image'])
            mask = transform(sample['mask'])
            mask = torch.where(mask > 0, 1.0, 0.0) 
            self.samples.append((image, mask))
            
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        return self.samples[idx]

print("Connecting to cloud dataset...")
dataset = load_dataset("sjyhne/mapai_dataset", split="train", streaming=True)
train_data = CachedBuildingDataset(dataset, num_samples=20)
train_loader = DataLoader(train_data, batch_size=4)

### 3. U-Net Architecture & Dice Loss
This section defines a lightweight **U-Net** architecture from scratch. 

**Handling Class Imbalance:** In satellite imagery, buildings typically account for less than 10% of pixels. Standard loss functions often lead to model collapse (predicting only background). To counter this, we implement a custom **DiceBCELoss** that heavily penalizes the model for failing to overlap with actual building structures.

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
    def forward(self, x): return self.conv(x)

class SimpleUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.down1 = DoubleConv(3, 32)
        self.pool1 = nn.MaxPool2d(2)
        self.down2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(64, 128)
        self.up1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.conv1 = DoubleConv(128, 64) 
        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.conv2 = DoubleConv(64, 32) 
        self.final_conv = nn.Conv2d(32, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid() 
        
    def forward(self, x):
        x1 = self.down1(x)
        x3 = self.down2(self.pool1(x1))
        b = self.bottleneck(self.pool2(x3))
        u1 = self.conv1(torch.cat([self.up1(b), x3], dim=1))
        u2 = self.conv2(torch.cat([self.up2(u1), x1], dim=1))
        return self.sigmoid(self.final_conv(u2))

class DiceBCELoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCELoss()
        
    def forward(self, inputs, targets, smooth=1):
        bce_loss = self.bce(inputs, targets)
        inputs_flat, targets_flat = inputs.view(-1), targets.view(-1)
        intersection = (inputs_flat * targets_flat).sum()
        dice_loss = 1.0 - ((2. * intersection + smooth) / (inputs_flat.sum() + targets_flat.sum() + smooth))
        return bce_loss + dice_loss

### 4. Model Training Engine
We train the network using the Adam optimizer. The loop calculates the loss, backpropagates the error, and updates the weights to incrementally improve feature detection.

In [ ]:
model = SimpleUNet()
criterion = DiceBCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 30
model.train()
print("Training U-Net on satellite imagery...")

for epoch in range(epochs):
    epoch_loss = 0.0
    for batch_images, batch_masks in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_images)
        loss = criterion(predictions, batch_masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Average Loss: {(epoch_loss / len(train_loader)):.4f}")

### 5. Inference & Adaptive Thresholding
To evaluate the model, we visualize the raw probability heatmap. Because raw confidence scores can be low during CPU-constrained training, we apply a statistical **adaptive threshold** (Mean + 1.5 Std Dev) to cleanly extract the final binary building footprints.

In [ ]:
images, masks = next(iter(train_loader))
test_image = images[0].unsqueeze(0)
true_mask = masks[0].squeeze()

model.eval()
with torch.no_grad():
    raw_prediction = model(test_image)
    
std_val = raw_prediction.std().item()
mean_val = raw_prediction.mean().item()
threshold = mean_val + (std_val * 1.5) if std_val > 0 else 0.5
adaptive_mask = (raw_prediction > threshold).float().squeeze()

fig, (ax1, ax2, ax3, ax4) = plt.subplots(1, 4, figsize=(20, 5))
ax1.imshow(test_image.squeeze().permute(1, 2, 0))
ax1.set_title("Original Image")
ax1.axis('off')
ax2.imshow(true_mask, cmap='gray')
ax2.set_title("Ground Truth")
ax2.axis('off')
im = ax3.imshow(raw_prediction.squeeze().numpy(), cmap='inferno')
ax3.set_title("Model Focus (Heatmap)")
ax3.axis('off')
fig.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)
ax4.imshow(adaptive_mask, cmap='gray')
ax4.set_title(f"Adaptive Mask")
ax4.axis('off')

plt.tight_layout()
plt.show()